In [10]:
import pandas as pd
import requests
import re
import html
import time
import random
from datetime import datetime, timezone

In [11]:
USER_AGENT = "bbc-notebook-review/1.0"

def now_utc():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sleep_polite():
    time.sleep(random.uniform(0.3, 0.8))


# Regex for extracting metadata
META_RE = re.compile(r"<meta\b[^>]*>", re.IGNORECASE)
ATTR_RE = re.compile(r'(\w[\w:-]*)\s*=\s*["\']([^"\']*)["\']')
TITLE_RE = re.compile(r"<title[^>]*>(.*?)</title>", re.IGNORECASE | re.DOTALL)


def parse_attrs(tag):
    return {k.lower(): html.unescape(v) for k, v in ATTR_RE.findall(tag)}


def extract(html_text):
    og_title = ""
    og_desc = ""
    meta_desc = ""
    title_tag = ""

    for tag in META_RE.findall(html_text):
        attrs = parse_attrs(tag)

        if attrs.get("property") == "og:title":
            og_title = attrs.get("content", "")

        if attrs.get("property") == "og:description":
            og_desc = attrs.get("content", "")

        if attrs.get("name") == "description":
            meta_desc = attrs.get("content", "")

    m = TITLE_RE.search(html_text)
    if m:
        title_tag = html.unescape(m.group(1))

    title = (og_title or title_tag).strip()
    desc = (og_desc or meta_desc).strip()

    # clean BBC suffix
    if title.lower().endswith(" - bbc news"):
        title = title[:-11].strip()

    return title, desc


In [12]:
df = pd.read_csv("output/bbc-output.csv")

# clean headers just in case
df.columns = df.columns.str.strip()

df.head()


,source_name,source_domain,url,published_at,retrieved_at,article_id,discovery_method,sitemap_url
0,BBC,bbc.com,https://www.bbc.com/news/world-middle-east-541...,2023-10-11T10:56:44Z,2026-02-18T15:17:26+00:00,54116567.0,bbc_xml_sitemap,https://www.bbc.com/sitemaps/https-sitemap-com...
1,BBC,bbc.com,https://www.bbc.com/news/world-middle-east-203...,2024-10-16T10:29:53Z,2026-02-18T15:17:37+00:00,20385306.0,bbc_xml_sitemap,https://www.bbc.com/sitemaps/https-sitemap-com...
2,BBC,bbc.com,https://www.bbc.com/news/world-middle-east-204...,2025-01-16T15:25:38Z,2026-02-18T15:17:37+00:00,20415675.0,bbc_xml_sitemap,https://www.bbc.com/sitemaps/https-sitemap-com...
3,BBC,bbc.com,https://www.bbc.com/news/world-middle-east-574...,2024-05-20T05:42:45Z,2026-02-18T15:17:40+00:00,57421235.0,bbc_xml_sitemap,https://www.bbc.com/sitemaps/https-sitemap-com...
4,BBC,bbc.com,https://www.bbc.com/news/world-middle-east-572...,2024-07-01T09:54:16Z,2026-02-18T15:18:03+00:00,57260831.0,bbc_xml_sitemap,https://www.bbc.com/sitemaps/https-sitemap-com...


In [13]:
for col in ["title", "description", "fetched_at", "http_status", "notes"]:
    if col not in df.columns:
        df[col] = ""

len(df)


1010

In [14]:
url = df.loc[0, "url"]

session = requests.Session()

r = session.get(url, headers={"User-Agent": USER_AGENT}, timeout=20)

title, desc = extract(r.text)

print(title)
print()
print(desc[:200])


Israel's borders explained in maps

The conflict between Israel and Palestinians has roots which precede the formation of the country itself. Here's how the shape of the Jewish state has changed.


In [20]:
session = requests.Session()

headers = {
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml",
}

N = 1000  # increase to 1000 when ready

ok = 0
fail = 0

for i in range(min(N, len(df))):

    url = str(df["url"].iloc[i]).strip()

    sleep_polite()

    try:
        r = session.get(url, headers=headers, timeout=20)

        df.loc[i, "http_status"] = r.status_code

        if r.status_code != 200:
            df.loc[i, "notes"] = f"non_200:{r.status_code}"
            fail += 1
            continue

        title, desc = extract(r.text)

        df.loc[i, "title"] = title
        df.loc[i, "description"] = desc
        df.loc[i, "fetched_at"] = now_utc()

        ok += 1

    except Exception as e:
        df.loc[i, "notes"] = str(type(e).__name__)
        fail += 1
        print("error:", url)

    if i % 10 == 0:
        print(f"{i}/{N} complete | ok={ok} fail={fail}")

print("DONE")


0/1000 complete | ok=1 fail=0
10/1000 complete | ok=11 fail=0
20/1000 complete | ok=21 fail=0
30/1000 complete | ok=31 fail=0
40/1000 complete | ok=41 fail=0
50/1000 complete | ok=51 fail=0
60/1000 complete | ok=61 fail=0
70/1000 complete | ok=71 fail=0
80/1000 complete | ok=81 fail=0
90/1000 complete | ok=91 fail=0
100/1000 complete | ok=101 fail=0
110/1000 complete | ok=111 fail=0
120/1000 complete | ok=121 fail=0
130/1000 complete | ok=131 fail=0
140/1000 complete | ok=141 fail=0
150/1000 complete | ok=151 fail=0
160/1000 complete | ok=161 fail=0
170/1000 complete | ok=171 fail=0
180/1000 complete | ok=181 fail=0
190/1000 complete | ok=191 fail=0
200/1000 complete | ok=201 fail=0
210/1000 complete | ok=211 fail=0
220/1000 complete | ok=221 fail=0
230/1000 complete | ok=231 fail=0
240/1000 complete | ok=241 fail=0
250/1000 complete | ok=251 fail=0
260/1000 complete | ok=261 fail=0
270/1000 complete | ok=271 fail=0
280/1000 complete | ok=281 fail=0
290/1000 complete | ok=291 fail=0
30

In [21]:
df[["url", "title", "description", "http_status", "notes"]].head(20)


,url,title,description,http_status,notes
0,https://www.bbc.com/news/world-middle-east-541...,Israel's borders explained in maps,The conflict between Israel and Palestinians h...,200,
1,https://www.bbc.com/news/world-middle-east-203...,"What are Israelâs Iron Dome, Davidâs Sling...",Israel has used its elaborate system of air de...,200,
2,https://www.bbc.com/news/world-middle-east-204...,Gaza war in maps and satellite images,A visual guide to the Gaza Strip since Israel ...,200,
3,https://www.bbc.com/news/world-middle-east-574...,Ebrahim Raisi: The hardline cleric who became ...,"The cleric, who became president in 2021, crac...",200,
4,https://www.bbc.com/news/world-middle-east-572...,Who is in charge of Iran?,How might the election of a new president affe...,200,
5,https://www.bbc.com/news/world-middle-east-146...,Kuwait country profile,"Provides an overview of Kuwait, including key ...",200,
6,https://www.bbc.com/news/world-middle-east-146...,Lebanon country profile,"Provides an overview of Lebanon, including key...",200,
7,https://www.bbc.com/news/world-middle-east-180...,"Benjamin Netanyahu, Israel's controversial leader",Israel's longest serving leader has faced tria...,200,
8,https://www.bbc.com/news/world-middle-east-147...,United Arab Emirates media guide,An overview of the media in the United Arab Em...,200,
9,https://www.bbc.com/news/world-middle-east-146...,Lebanon media guide,"An overview of the media in Lebanon, including...",200,


In [22]:
df.to_csv("output/bbc-reviewed.csv", index=False)

print("Saved to output/bbc-reviewed.csv")


Saved to output/bbc-reviewed.csv


In [18]:
N = 1000
